In [ ]:
!pip install duckdb
!pip install pyarrow

In [1]:
from pathlib import Path
import geopandas as gpd
import duckdb
import pyarrow.parquet as pq
import pyarrow as pa

In [2]:
# Constants
RELEASE = "2025-11-19.0"  # OpenStreetMap data release date

data_inputs = Path("../scripts/Beira/data-inputs/").resolve()
data_inputs.mkdir(parents=True, exist_ok=True)

boundary_file = data_inputs / "grid-boundary-beira.gpkg"

out_bbox_parquet = data_inputs / "Building-footprint.parquet"
out_clip_parquet = data_inputs / "Building-footprint-clipped.parquet"
out_clip_geojson = data_inputs / "Building-footprint-clipped.geojson"


In [3]:
# ===== 1) boundary + bbox (EPSG:4326) =====
gdf = gpd.read_file(boundary_file)
study_area = gdf.dissolve().reset_index(drop=True)

study_area_4326 = study_area.to_crs(4326)
minx, miny, maxx, maxy = study_area_4326.total_bounds
print("bbox:", minx, miny, maxx, maxy)


bbox: 34.788621889927356 -19.862655260191364 34.917123981550304 -19.705375297473417


In [4]:
# ===== 2) DuckDB read from public S3, filter by bbox struct =====
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")

src = f"s3://overturemaps-us-west-2/release/{RELEASE}/theme=buildings/type=building/*.parquet"

query = f"""
SELECT id, geometry, bbox
FROM read_parquet('{src}', filename=true, hive_partitioning=1)
WHERE
  bbox.xmin <= {maxx} AND bbox.xmax >= {minx}
  AND bbox.ymin <= {maxy} AND bbox.ymax >= {miny}
"""

tbl = con.execute(query).fetch_arrow_table()
df = tbl.to_pandas()

# ===== convert to GeoDataFrame =====
buildings_4326 = gpd.GeoDataFrame(
    df.drop(columns=["bbox"]),
    geometry=gpd.GeoSeries.from_wkb(df["geometry"]),
    crs=4326
)
print("downloaded rows:", len(buildings_4326))

# ===== save bbox parquet =====
buildings_4326.to_parquet(out_bbox_parquet)
print("Saved bbox parquet:", out_bbox_parquet)

downloaded rows: 225634
Saved bbox parquet: /Users/moriseiitsu/Dropbox/IDEAMAPS/ideamaps-models/models/emergency-maternal-care/scripts/Beira/data-inputs/Building-footprint.parquet


In [5]:
# ===== 3) clip to study area =====
buildings = buildings_4326.to_crs(study_area.crs)
clipped = gpd.clip(buildings, study_area)

clipped.to_parquet(out_clip_parquet)
clipped.to_crs(4326).to_file(out_clip_geojson, driver="GeoJSON")

print("Saved clipped parquet:", out_clip_parquet)
print("Saved clipped geojson:", out_clip_geojson)

Saved clipped parquet: /Users/moriseiitsu/Dropbox/IDEAMAPS/ideamaps-models/models/emergency-maternal-care/scripts/Beira/data-inputs/Building-footprint-clipped.parquet
Saved clipped geojson: /Users/moriseiitsu/Dropbox/IDEAMAPS/ideamaps-models/models/emergency-maternal-care/scripts/Beira/data-inputs/Building-footprint-clipped.geojson
